# Module 3: Localisation and synthesis

Takes the defect bank from Module 2 and composites those defects onto new defect-free images,
producing synthetic anomalies with pixel-accurate masks. No image generation model is used.

## Requirements

Kaggle notebook, accelerator set to GPU T4 x2. No language or vision model is needed.

Attach two mounts:

1. Your Module 2 defect bank, as a Dataset.
2. MVTec AD 2, as a Dataset, for the host images.

## Steps

1. Open section 1 and set the mount paths, `N_SYNTH` and `SEEDS`. Corpus size is
   `categories x SEEDS x N_SYNTH`.
2. Run all cells in order. Start with `N_SYNTH = 1` to check the setup before committing time.
3. Check the `replay@1024` column printed in section 2. It is the size each defect will appear at.
   If those numbers look wrong, the bank manifest is wrong.
4. Check the sanity figure in section 4, which shows the detected object region and the placement
   blob for one host per category.
5. Read the summary in section 7 and look at the images in section 9.

## Output

```
Final Synthetic Images/
  <category>/<seed>/<n>_<entry>.png          the synthetic anomaly
  <category>/<seed>/<n>_<entry>_mask.png     its ground-truth mask
  manifest.csv                               one row per image
  run_config.json                            the settings used
```

## 1. Config

Set the mount paths, corpus size and output directory here. The values below are the published
configuration and are applied unchanged to every category.

| setting | value | controls |
|---|---|---|
| `WORK_SIZE` | 1024 | resolution everything is composited at |
| `AD2_COVERAGE` | 0.018 | placement blob size, as a fraction of the detected object |
| `COLLAR_FRAC` | 0.25 | width of the substrate margin given to the Poisson solve |
| `PLACEMENT_MIX` | 0.5 | fraction of samples using adaptive rather than original placement |
| `R_EQ_POISSON` | 16.0 | defects smaller than this radius composite with alpha instead |
| `N_SYNTH`, `SEEDS` | | corpus size |

Pixel-denominated values are tied to `WORK_SIZE`. If you change the resolution, rescale them.

In [ ]:
import os, glob, math, json, random, warnings, time, zlib, shutil, csv, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import cv2
from PIL import Image
from scipy import ndimage
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(SEED)
def IMAGE_TO_TENSOR(im):
    """PIL RGB -> C x H x W float tensor in [0,1].

    This is all torchvision's ToTensor did here, and importing torchvision for one call makes
    the notebook fail on any environment that has torch but not torchvision.
    """
    a = np.asarray(im, dtype=np.float32) / 255.0
    return torch.from_numpy(a).permute(2, 0, 1).contiguous()

N_CPU = max(1, (os.cpu_count() or 2))
print(f"torch {torch.__version__} | cv2 {cv2.__version__} | seed {SEED}")
print(f"cuda: {torch.cuda.is_available()} "
      f"({torch.cuda.device_count()} device(s))" if torch.cuda.is_available() else "cuda: no")
print(f"cpu workers available: {N_CPU}")

In [ ]:
# --- Configuration ==================================================================
# Used only to SCORE a candidate MVTec root during discovery. The categories actually
# processed are whatever Local Output contains -- see CATS_SEEN in section 2.
CATS  = ["can", "fabric", "fruit_jelly", "rice",
         "sheet_metal", "vial", "wallplugs", "walnuts"]
SEEDS = [0, 1, 2]

# HOW MANY IMAGES. This is the knob you actually turn.
# Corpus size = len(LIVE_CATS) x len(SEEDS) x N_SYNTH.
#   proof of concept   1  ->  8 x 3 x 1 =  24 images     <- current
#   small run          4  ->  8 x 3 x 4 =  96
#   full run          24  ->  8 x 3 x 24 = 576
# All three seeds are kept even at PoC size: the seed is what varies the blob, the bank entry
# and the rotation, so dropping seeds would hide exactly the variation this is meant to show.
N_SYNTH = 1
N_HOSTS = 32        # ceiling on the host pool per category

WORK_SIZE = 1024    # synthesis resolution (paper Table 3)

# --- MRSP + OBS (paper 3.4) ---------------------------------------------------------
AD2_NOISE_SCALE = 14
OCTAVES_FBM     = 6
PERSISTENCE     = 0.8
MRSP_ALPHA      = 1.5     # 1/f^alpha spectral decay -- NOT in the paper's Table 3; it should be
SINGLE_BLOB     = True
MIN_BLOB_PX     = 25
AD2_COVERAGE    = 0.018

# Empty-frame gate. Relative, not absolute: across eight categories normal OBS coverage runs
# from a few scattered wallplugs to a fabric sheet filling the frame, so one fixed fraction
# would either pass every empty walnuts tray or reject every good vial.
MIN_OBJ_COV_REL = 0.40
MIN_OBJ_COV_ABS = 0.02

# --- Placement (paper 3.5, Algorithm 4) ---------------------------------------------
PLACE_MODE    = "real_scale"   # the defect replays at its TRUE relative size
ROTATE        = True
PLACEMENT_MIX = 0.5            # fraction of samples generated by mode B
GAMMA_RANGE   = (0.20, 0.60)   # defect's share of the blob, drawn per sample
GAMMA_S_RANGE = (0.15, 2.50)
CONTAIN_T     = 0.90
R_EQ_POISSON  = 16.0           # below this, route to alpha

# --- Compositing --------------------------------------------------------------------
BETA          = 1.0
FEATHER_PX    = 1.5
COLLAR_FRAC   = 0.25
COLLAR_MIN_PX = 6
COLLAR_KEEP_R = 0.5
DEFECT_T      = 6.0

# A composite whose GT region differs from the host by less than this (mean |out-host| over the
# GT, 0-255) carries no findable defect. 8/255 is ~3% mean channel change: below that the edit
# sits at sensor-noise level. It is a REPORTING threshold -- nothing is dropped. Poisson erasing
# a small defect is a property of the operator, not a bug, and removing those samples would
# quietly flatter it.
DISSOLVED_T   = 8.0

# --- Bank ---------------------------------------------------------------------------
# LOADED, not built. Stages 2-3 extracted and validated these crops; this notebook spends them.
# That split is the "generate once, synthesise many" architecture -- re-extracting here would
# bypass the VLM gate and silently put rejected crops back into the corpus.
MIN_ENTRY_CFRAC = 0.10   # re-applied on load as a guard; the bank already passed it

# Only used when the bank ships WITHOUT a manifest. defect_frac = defect_long_px / this, and it
# is the one field that cannot be recovered from a crop PNG: a 322px defect from a 2448px frame
# and the same 322px from a 1024px frame are different sizes relative to the object. Wrong here
# means every defect is placed at the wrong scale, so the notebook says loudly when it is used.
SOURCE_LONG_PX = 2448    # MVTec AD 2 native long side

DEFECT_KIND = {"rice": "foreign_object",   "walnuts": "surface",
               "wallplugs": "foreign_object", "fruit_jelly": "foreign_object",
               "can": "surface",           "fabric": "surface",
               "sheet_metal": "surface",   "vial": "foreign_object"}

# --- Output -------------------------------------------------------------------------
OUT_ROOT  = "/kaggle/working" if os.path.isdir("/kaggle/working") else "out_stage45"
FINAL_DIR = os.path.join(OUT_ROOT, "Final Synthetic Images")
os.makedirs(OUT_ROOT, exist_ok=True)

# Dataset ref OR literal path. A kaggle URL and an "owner/slug" both work: mount_candidates()
# in section 2 expands them, because Kaggle mounts the same dataset at different paths on
# different kernels (/kaggle/input/<slug> vs /kaggle/input/datasets/<owner>/<slug>).
BANK_ID = "/kaggle/input/datasets/abhaykdas/defect-bank"        # Stage 2-3 output
AD2_ID  = "/kaggle/input/datasets/abhaykdas/mvtec-ad2-reducded"  # normals for hosts

print(f"target: {len(SEEDS)} seeds x {N_SYNTH} per category "
      f"(x however many categories the bank has)")
print(f"output -> {FINAL_DIR}")

## 2. Inputs

Locates the two mounts and loads the bank.

Expected bank layout:

```
<bank>/manifest.csv
<bank>/<category>/<key>.png
<bank>/<category>/<key>_alpha.png
```

Both image files are required. The crop keeps its own background, which the harmoniser measures,
and the mask becomes the ground truth.

`manifest.csv` supplies `defect_frac`, the defect's size relative to its original frame. It cannot
be recovered from the PNG. If the manifest is missing, the notebook falls back to an assumed source
resolution and says so; check the `replay@1024` column before trusting the run.

Host images come from `<mvtec>/<category>/train/good`. Donor images are removed from the host pool,
and the number dropped is printed. If that count is zero, donors and hosts are not being matched
and the same frame may appear on both sides.

In [ ]:
_IMG_EXT = ("png", "PNG", "jpg", "JPG", "jpeg", "JPEG")


def mount_candidates(hint):
    """Turn a dataset reference into the paths Kaggle might actually have mounted it at.

    Accepts a full URL, an "owner/slug" ref, or a literal path. Kaggle is not consistent about
    the mount point -- the same dataset appears as /kaggle/input/<slug> on some kernels and
    /kaggle/input/datasets/<owner>/<slug> on others -- so every plausible form is tried.
    """
    if not hint:
        return []
    h = hint.strip().rstrip("/\\")
    if os.path.isdir(h):                 # before the "/" test: a Windows path is absolute
        return [h]                       # without starting with "/"
    if h.startswith("http"):
        parts = [x for x in h.split("/") if x]
        if "datasets" in parts:
            k = parts.index("datasets")
            h = "/".join(parts[k + 1:k + 3])
    if os.path.isabs(h):
        # An absolute path that does not exist is still a strong hint about the slug, so fall
        # through to the slug forms rather than returning a path that is known to be missing.
        h = "/".join([p for p in h.split("/") if p][-2:])
    bits = [b for b in h.split("/") if b]
    slug, owner = bits[-1], (bits[-2] if len(bits) >= 2 else None)
    cands = [f"/kaggle/input/{slug}"]
    if owner:
        cands += [f"/kaggle/input/{owner}/{slug}", f"/kaggle/input/datasets/{owner}/{slug}"]
    return cands


def find_root(pred, hint, what, max_depth=8):
    """`hint` in any mount form if it satisfies `pred`, else the BEST match under /kaggle/input.

    Best, not first: a previous run's output can look like the real thing from the outside, and
    returning the first hit is how you end up synthesising from the wrong folder.
    """
    for cand in mount_candidates(hint):
        try:
            if os.path.isdir(cand) and pred(cand):
                print(f"  {what}: {cand}")
                return cand
        except OSError:
            pass
    if hint:
        print(f"  {what}: no match at any mount form of {hint!r} -- searching")
    base = "/kaggle/input" if os.path.isdir("/kaggle/input") else "."
    best, best_n = None, 0
    for root, dirs, _ in os.walk(base):
        if root.count(os.sep) - base.count(os.sep) > max_depth:
            dirs[:] = []; continue
        dirs[:] = [d for d in dirs if not d.startswith(".")
                   and not glob.glob(os.path.join(root, d, "*.safetensors"))]
        try:
            n = pred(root)
        except OSError:
            continue
        if n and n > best_n:
            best, best_n = root, int(n)
    if best is None:
        raise FileNotFoundError(f"could not locate {what} under {base}")
    print(f"  {what}: {best}  (score {best_n})")
    return best


def _score_bank(d):
    """Categories under `d` that hold at least one <key>.png + <key>_alpha.png pair.

    Scored on the images, not on manifest.csv, so a bank uploaded without one is still found --
    and so a stray manifest elsewhere in the tree cannot win.
    """
    n = 0
    for c in sorted(os.listdir(d)):
        p = os.path.join(d, c)
        if os.path.isdir(p) and glob.glob(os.path.join(p, "*_alpha.png")):
            n += 1
    return n

def _score_ad2(d):
    return sum(os.path.isdir(os.path.join(d, c, "train", "good")) for c in CATS)


BANK_ROOT = find_root(_score_bank, BANK_ID, "defect bank")
AD2_ROOT  = find_root(_score_ad2, AD2_ID, "MVTec AD 2 reduced")

In [ ]:
# --- Loaders ------------------------------------------------------------------------
def load_image(path, size=WORK_SIZE):
    """Longest side -> size (aspect preserved) -> C x H x W float in [0,1]."""
    im = Image.open(path).convert("RGB")
    w, h = im.size
    s = size / max(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
    return IMAGE_TO_TENSOR(im)

def as_np(t):
    return t.permute(1, 2, 0).numpy() if t.dim() == 3 else t.numpy()

def _imgs_in(d):
    out = []
    if os.path.isdir(d):
        for ext in _IMG_EXT:
            out += glob.glob(os.path.join(d, f"*.{ext}"))
    # set(): on a case-insensitive filesystem "*.png" and "*.PNG" both match every file, which
    # silently doubles the pool -- the same image becomes two hosts and the audit counts lie.
    return sorted(set(out))


def load_bank(root):
    """Read the Stage 2-3 bank. rgb and alpha stay SEPARATE, never merged to RGBA: harmonise()
    measures the crop's own substrate, and an RGBA cut-out has none.

    Driven by manifest.csv when present. Without it, side and defect_long_px are measured from
    the images and defect_frac is derived from SOURCE_LONG_PX -- which is an assumption, and is
    announced as one.
    """
    import csv as _csv
    mpath = os.path.join(root, "manifest.csv")
    meta = {}
    if os.path.exists(mpath):
        with open(mpath, newline="", encoding="utf-8") as fh:
            for r in _csv.DictReader(fh):
                meta[r["key"]] = r
        print(f"  manifest.csv: {len(meta)} rows")
    else:
        print("  NO manifest.csv -- geometry measured from the images and defect_frac derived")
        print(f"     from SOURCE_LONG_PX={SOURCE_LONG_PX}. If the crops did not come from "
              f"{SOURCE_LONG_PX}px")
        print("     frames, every defect will be placed at the wrong scale.")

    bank, skipped = {}, []
    for ap in sorted(glob.glob(os.path.join(root, "*", "*_alpha.png"))):
        cat = os.path.basename(os.path.dirname(ap))
        key = os.path.basename(ap)[: -len("_alpha.png")]
        rp = os.path.join(os.path.dirname(ap), f"{key}.png")
        rgb = cv2.imread(rp, cv2.IMREAD_COLOR)
        alpha = cv2.imread(ap, cv2.IMREAD_GRAYSCALE)
        if rgb is None or alpha is None:
            skipped.append((key, "missing rgb or alpha")); continue

        m = meta.get(key)
        if m:
            if float(m.get("cfrac", 0) or 0) < MIN_ENTRY_CFRAC:
                skipped.append((key, f"cfrac {m.get('cfrac')} < {MIN_ENTRY_CFRAC}")); continue
            e = dict(m)
            e["side"] = int(float(m["side"]))
            e["defect_long_px"] = int(float(m["defect_long_px"]))
            e["defect_frac"] = float(m["defect_frac"])
            e["cat"] = m.get("cat") or cat
        else:
            ys, xs = np.nonzero(alpha > 127)
            if ys.size == 0:
                skipped.append((key, "empty alpha")); continue
            long_px = int(max(ys.max() - ys.min() + 1, xs.max() - xs.min() + 1))
            e = dict(key=key, cat=cat, donor=key.rsplit("_", 2)[0],
                     kind=DEFECT_KIND.get(cat, "unknown"), defect_type="unknown")
            e["side"] = int(rgb.shape[0])
            e["defect_long_px"] = long_px
            e["defect_frac"] = long_px / float(SOURCE_LONG_PX)

        e["rgb"] = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        e["alpha"] = alpha.astype(np.float32) / 255.0
        bank.setdefault(e["cat"], []).append(e)

    if skipped:
        print(f"  {len(skipped)} entr(y/ies) skipped: "
              + "; ".join(f"{k} ({w})" for k, w in skipped[:5])
              + (" ..." if len(skipped) > 5 else ""))
    return bank


BANK = load_bank(BANK_ROOT)
CATS_SEEN = sorted(BANK)
LIVE_CATS = [c for c in CATS_SEEN if BANK[c]]
DEAD_CATS = [c for c in CATS_SEEN if not BANK[c]]
if not LIVE_CATS:
    raise SystemExit(f"no usable bank entries under {BANK_ROOT}")

print(f"\n{sum(len(v) for v in BANK.values())} entries across {len(LIVE_CATS)} categories\n")
print(f"{'entry key':32s} {'cat':13s} {'crop':>8s} {'defect':>8s} {'frac':>8s}"
      f" {'replay@1024':>12s} {'subst%':>7s}")
for c in LIVE_CATS:
    for e in BANK[c]:
        print(f"{e['key']:32s} {e['cat']:13s} {e['side']:6d}px {e['defect_long_px']:6d}px"
              f" {e['defect_frac']:8.4f} {e['defect_frac']*WORK_SIZE:10.0f}px"
              f" {100*(e['alpha']<0.1).mean():6.1f}%")
if DEAD_CATS:
    print(f"\nDROPPED (no usable entry): {', '.join(DEAD_CATS)}")

In [ ]:
# --- Host pool: donors removed -------------------------------------------------------
# The manifest stores donor as "<cat>-<id>" ("fruit_jelly-064") because ids collide across
# categories, but the normals on disk are named "064_regular.png". Comparing the two directly
# never matches, so the prefix comes off first -- and the drop count is printed, because a
# filter that silently removes nothing is worse than no filter.
HOST_POOL = {}
for c in LIVE_CATS:
    donors = set()
    for e in BANK[c]:
        d = str(e.get("donor", ""))
        donors.add(d[len(c) + 1:] if d.startswith(c + "-") else d)
    stems = set()
    for i in donors:
        stems |= {i, f"{i}_regular", (i.lstrip("0") or "0")}

    pool = _imgs_in(os.path.join(AD2_ROOT, c, "train", "good"))
    kept = [p for p in pool if os.path.splitext(os.path.basename(p))[0] not in stems]
    n_drop = len(pool) - len(kept)
    HOST_POOL[c] = kept[:N_HOSTS]
    print(f"  {c:14s} {len(pool):3d} normals - {n_drop} donor(s) = {len(kept):3d}"
          f" -> {len(HOST_POOL[c])} hosts")
    if n_drop == 0 and donors:
        print(f"      WARNING: no donor matched a filename in train/good "
              f"(ids {sorted(donors)[:3]}) -- a donor may also be used as a host")
    if len(HOST_POOL[c]) < N_SYNTH:
        print(f"      WARNING: fewer hosts than N_SYNTH={N_SYNTH}; hosts will be reused")

## 3. Object detection and placement field

Two operators run per host image.

Object Boundary Suppression estimates the region the product occupies, using colour distance from
the border-median background combined with a local texture measure. Either cue alone fails on some
categories, so a pixel is kept when either fires. If the result covers almost none or almost all of
the frame, it falls back to the whole frame.

The placement field is multi-resolution spectral noise, thresholded to the target coverage. The
threshold is computed from the field values inside the detected object only, which is what makes
the coverage relative to the product rather than the frame.

An empty-frame gate then removes hosts whose detected object is far below the category median.
Those are frames where the product is absent, and a defect placed on one is a nonsense sample.

In [ ]:
# --- MRSP: multi-resolution spectral pyramid noise (from flash_part1) ---------------
def generate_mrsp(h, w, scale=8, device="cpu", seed=None, alpha=1.5,
                  levels=OCTAVES_FBM, persistence=PERSISTENCE):
    if seed is not None:
        torch.manual_seed(int(seed))
    device = torch.device(device)
    field = torch.zeros(h, w, device=device, dtype=torch.float32)
    weight_sq_sum = 0.0
    for level in range(levels):
        factor = 2 ** (levels - 1 - level)
        h_l = max(8, h // factor); w_l = max(8, w // factor)
        fy = torch.fft.fftfreq(h_l, device=device).reshape(-1, 1)
        fx = torch.fft.rfftfreq(w_l, device=device).reshape(1, -1)
        f = torch.sqrt(fy * fy + fx * fx) * (1.0 + math.log2(max(1, int(scale))))
        amp = torch.where(f > 1e-6, f.pow(-alpha), torch.zeros_like(f))
        phase = 2 * math.pi * torch.rand(h_l, w_l // 2 + 1, device=device)
        spec = amp * torch.complex(torch.cos(phase), torch.sin(phase))
        noise_lr = torch.fft.irfft2(spec, s=(h_l, w_l))
        if (h_l, w_l) != (h, w):
            noise = F.interpolate(noise_lr.unsqueeze(0).unsqueeze(0), size=(h, w),
                                  mode="bilinear", align_corners=False).squeeze()
        else:
            noise = noise_lr
        noise = (noise - noise.mean()) / (noise.std() + 1e-8)
        w_oct = persistence ** level
        field = field + w_oct * noise
        weight_sq_sum += w_oct ** 2
    field = field / math.sqrt(weight_sq_sum + 1e-8)
    return (field - field.mean()) / (field.std() + 1e-8)


# --- OBS: object-aware foreground (from flash_part1) --------------------------------
def _fg_robust(image, use_cues=True):
    img = (np.clip(image.permute(1, 2, 0).numpy(), 0, 1) * 255).astype(np.uint8)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
    border = np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]], 0)
    bg = np.median(border, 0)
    dist = np.sqrt(((lab - bg) ** 2).sum(2))
    du8 = (255 * dist / (dist.max() + 1e-8)).astype(np.uint8)
    _, cm = cv2.threshold(du8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cm > 0
    if use_cues:
        gray = img.mean(2).astype(np.float32) / 255.0
        k = max(15, round(min(img.shape[:2]) / 64))
        mean = ndimage.uniform_filter(gray, k)
        sq = ndimage.uniform_filter(gray * gray, k)
        std = np.sqrt(np.clip(sq - mean * mean, 0, None))
        su8 = (255 * std / (std.max() + 1e-8)).astype(np.uint8)
        _, tm = cv2.threshold(su8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        m = m | (tm > 0)
    m = ndimage.binary_opening(m, iterations=1)
    m = ndimage.binary_closing(m, iterations=2)
    m = ndimage.binary_fill_holes(m)
    lbl, n = ndimage.label(m)
    if n:
        sz = ndimage.sum(np.ones_like(lbl, dtype=np.float32), lbl, range(1, n + 1))
        keep = np.where(sz >= 0.0015 * m.size)[0] + 1
        if len(keep) == 0:
            keep = [int(sz.argmax()) + 1]
        m = np.isin(lbl, keep)
    return m.astype(np.float32)


def region_for(image):
    """Algorithm 2 step 11 -- the degeneracy guard."""
    region = _fg_robust(image, use_cues=True); cov = region.mean()
    if cov < 0.03 or cov > 0.99:
        region = np.ones_like(region)
    return torch.from_numpy(region).float()


def coverage_mask(noise, region, target_coverage):
    region = region.to(noise.device)
    values = noise[region > 0]
    if values.numel() == 0:
        return torch.zeros_like(noise)
    thr = torch.quantile(values, 1.0 - target_coverage)
    return ((noise > thr) & (region > 0)).float()


def mrsp_blob_stats(mask, min_blob_px=MIN_BLOB_PX):
    """Area-weighted characteristic blob side + largest-blob centroid."""
    lbl, n = ndimage.label(mask.numpy() > 0.5)
    if n == 0:
        return 0.0, None
    areas = ndimage.sum(np.ones_like(lbl, dtype=np.float32), lbl, range(1, n + 1))
    big = areas[areas >= min_blob_px]
    if big.size == 0:
        big = areas
    char_side = math.sqrt(float((big ** 2).sum() / big.sum()))
    i = int(areas.argmax()) + 1
    ys, xs = np.where(lbl == i)
    return char_side, (int(ys.mean()), int(xs.mean()))


def keep_largest_blob(mask):
    lbl, k = ndimage.label(mask.numpy() > 0.5)
    if k <= 1:
        return mask
    sizes = ndimage.sum(np.ones_like(lbl, np.float32), lbl, range(1, k + 1))
    return torch.from_numpy((lbl == int(sizes.argmax()) + 1).astype(np.float32))


MRSP_DEVICE = "cpu"          # replaced by the measurement in section 4

def mrsp_mask_for(img, seed=SEED, region=None, coverage=None):
    _, H, W = img.shape
    if region is None:
        region = region_for(img)
    noise = generate_mrsp(H, W, scale=AD2_NOISE_SCALE, device=MRSP_DEVICE, seed=seed,
                          alpha=MRSP_ALPHA).cpu()
    mask = coverage_mask(noise, region, AD2_COVERAGE if coverage is None else coverage)
    if SINGLE_BLOB:
        mask = keep_largest_blob(mask)
    return region, mask


# OBS is deterministic per host and is recomputed for every seed unless cached. At 1024px it is
# ~0.2s of morphology; three seeds over 24 hosts is 72 recomputations of the same answer.
_HOST_CACHE = {}
def host_and_region(path):
    key = (path, WORK_SIZE)
    if key not in _HOST_CACHE:
        img = load_image(path)
        _HOST_CACHE[key] = (img, region_for(img))
    img, reg = _HOST_CACHE[key]
    return img.clone(), reg

print("MRSP + OBS ready.")

In [ ]:
# --- Empty-frame gate, measured per category ---------------------------------------
# On an empty frame OBS returns the tray, so the placement is on-region and still nonsense
# (a shell cavity floating on a conveyor). The gate is RELATIVE to the category's own median.
MIN_OBJ_COV_CAT, HOST_PATHS = {}, {}
_t0 = time.time()
for c in LIVE_CATS:
    covs = [(hp, float(host_and_region(hp)[1].mean())) for hp in HOST_POOL[c]]
    med = float(np.median([v for _, v in covs])) if covs else 0.0
    thr = max(MIN_OBJ_COV_ABS, MIN_OBJ_COV_REL * med)
    MIN_OBJ_COV_CAT[c] = thr
    HOST_PATHS[c] = [hp for hp, v in covs if v >= thr]
    dead = [(hp, v) for hp, v in covs if v < thr]
    print(f"  {c:14s} median cov {med:.3f}  gate {thr:.3f}  -> {len(HOST_PATHS[c])} hosts"
          + (f"  (dropped {len(dead)} empty)" if dead else ""))
print(f"\nOBS over {sum(len(v) for v in HOST_POOL.values())} hosts in {time.time()-_t0:.1f}s "
      f"(cached; not recomputed per seed)")

## 4. Device selection

Times the noise generation on CPU and on GPU and uses whichever is faster.

Only the noise field benefits from the GPU. Object detection and compositing are OpenCV
operations, and `cv2.seamlessClone` in particular is CPU only and is the most expensive step in
this notebook. Do not expect the accelerator to change the total runtime much.

The sanity figure below shows one host per category with the detected object region and the
placement blob drawn on it. Check that the blob sits on the product.

In [ ]:
def _bench_mrsp(device, n=6, h=WORK_SIZE, w=WORK_SIZE):
    try:
        generate_mrsp(h, w, scale=AD2_NOISE_SCALE, device=device, seed=0,
                      alpha=MRSP_ALPHA)                     # warmup / autotune
        if device != "cpu":
            torch.cuda.synchronize()
        t0 = time.time()
        for i in range(n):
            generate_mrsp(h, w, scale=AD2_NOISE_SCALE, device=device,
                          seed=i, alpha=MRSP_ALPHA).cpu()
        if device != "cpu":
            torch.cuda.synchronize()
        return (time.time() - t0) / n
    except Exception as e:
        print(f"  {device}: unavailable ({type(e).__name__})")
        return float("inf")

# Synthesis runs SERIALLY, exactly as flash_part1 does it. An earlier version fanned out over
# ProcessPoolExecutor and deadlocked: the pool forks, a CUDA context does not survive fork(),
# and every worker hung re-initialising CUDA on its first job. Serial removes that whole class
# of bug and lets MRSP use the GPU safely, which is the better trade at this corpus size.
_cpu = _bench_mrsp("cpu")
print(f"  cpu   {_cpu*1000:7.1f} ms/field")
if torch.cuda.is_available():
    _gpu = _bench_mrsp("cuda:0")
    print(f"  cuda  {_gpu*1000:7.1f} ms/field")
    MRSP_DEVICE = "cuda:0" if _gpu < _cpu else "cpu"
    print(f"\nMRSP -> {MRSP_DEVICE} ({max(_cpu,_gpu)/max(min(_cpu,_gpu),1e-9):.1f}x faster)")
else:
    MRSP_DEVICE = "cpu"
    print("\nMRSP -> cpu (no cuda)")

print("compositing -> serial; cv2.seamlessClone has no GPU path and forking a CUDA "
      "context deadlocks")

In [ ]:
# --- Sanity: host -> OBS region -> MRSP blob ----------------------------------------
_show = LIVE_CATS[:4]
fig, ax = plt.subplots(len(_show), 3, figsize=(13, 4.3 * len(_show)), squeeze=False)
for r, cat in enumerate(_show):
    img, reg = host_and_region(HOST_PATHS[cat][0])
    _, mask = mrsp_mask_for(img, seed=SEED, region=reg)
    base = as_np(img)
    ov_r = base.copy(); ov_r[..., 0] = np.maximum(ov_r[..., 0], 0.45 * reg.numpy())
    ov_m = base.copy(); ov_m[..., 1] = np.maximum(ov_m[..., 1], 0.85 * mask.numpy())
    side, _ = mrsp_blob_stats(mask)
    ax[r, 0].imshow(base); ax[r, 0].set_title(f"{cat} - host normal", fontsize=9)
    ax[r, 1].imshow(ov_r); ax[r, 1].set_title(f"Ω  ({100*reg.mean():.0f}% of frame)", fontsize=9)
    ax[r, 2].imshow(ov_m); ax[r, 2].set_title(f"M_f - char side {side:.0f}px", fontsize=9)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout(); plt.show()      # shown, not saved: the output dir holds the corpus only

## 5. Placement and blending

Two placement modes are used, mixed according to `PLACEMENT_MIX`.

Original placement puts the defect at the centre of the placement blob, at its true size relative
to the frame, with a random rotation.

Adaptive placement additionally scales the defect to a fraction of the blob area, aligns its
principal axis to the blob's, anchors it by centre of mass, and shrinks it until it sits inside the
blob. The defect is only ever scaled by the same factor in both directions, so its shape is
preserved.

Harmonisation then transfers CIELAB statistics from the crop's own background to the host's local
background, leaving the defect pixels untouched.

Blending uses Poisson cloning with a substrate collar: the mask is widened before the solve so the
boundary sits on clean product surface rather than through the defect. Defects below
`R_EQ_POISSON` cannot spare that margin and composite with alpha instead.

In [ ]:
# --- Placement (verbatim from flash_part1) ------------------------------------------
def target_long_px(entry, H, W):
    """real_scale: the defect replays at its true relative size."""
    return entry["defect_frac"] * max(H, W)

def place_entry(host, entry, center, long_px, theta_deg, flip, region):
    """Resize + optional flip/rotate the entry, stamp into a host-sized canvas initialised
    to the host, so the patch's own substrate is present for the band solve to work against."""
    _, H, W = host.shape
    rgb, a = entry["rgb"], entry["alpha"]
    if flip:
        rgb, a = rgb[:, ::-1].copy(), a[:, ::-1].copy()

    r = long_px / max(entry["defect_long_px"], 1)
    s = int(np.clip(round(entry["side"] * r), 8, max(H, W)))
    interp = cv2.INTER_AREA if r < 1.0 else cv2.INTER_LANCZOS4
    rgb = cv2.resize(rgb, (s, s), interpolation=interp)
    a   = cv2.resize(a,   (s, s), interpolation=cv2.INTER_LINEAR)

    if theta_deg:
        p = int(math.ceil(s * (math.sqrt(2) - 1) / 2)) + 2
        rgb = cv2.copyMakeBorder(rgb, p, p, p, p, cv2.BORDER_REFLECT_101)
        a   = cv2.copyMakeBorder(a,   p, p, p, p, cv2.BORDER_CONSTANT, value=0)
        s2 = rgb.shape[0]
        M = cv2.getRotationMatrix2D((s2 / 2, s2 / 2), theta_deg, 1.0)
        rgb = cv2.warpAffine(rgb, M, (s2, s2), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        a   = cv2.warpAffine(a,   M, (s2, s2), flags=cv2.INTER_LINEAR, borderValue=0)
        s = s2

    cy, cx = center
    y0 = int(np.clip(cy - s // 2, 0, max(0, H - s))); x0 = int(np.clip(cx - s // 2, 0, max(0, W - s)))
    y1, x1 = min(H, y0 + s), min(W, x0 + s)
    sy, sx = y1 - y0, x1 - x0

    patch = host.clone()
    alpha = torch.zeros(H, W)
    pt = torch.from_numpy(rgb[:sy, :sx].astype(np.float32) / 255.0).permute(2, 0, 1)
    patch[:, y0:y1, x0:x1] = pt
    alpha[y0:y1, x0:x1] = torch.from_numpy(np.clip(a[:sy, :sx], 0, 1).astype(np.float32))
    alpha = alpha * region                     # never place off-object
    return patch, alpha, (y0, x0, sy, sx)


# --- Harmonisation (verbatim from flash_part1) --------------------------------------
def harmonise(patch, alpha, host, box, std_clip=(0.75, 1.35)):
    y0, x0, sy, sx = box
    _, H, W = host.shape
    a = alpha.numpy()
    src_sel = np.zeros((H, W), bool); src_sel[y0:y0 + sy, x0:x0 + sx] = True
    src_sel &= (a < 0.10)
    pad = int(0.6 * max(sy, sx)) + 8
    ry0, ry1 = max(0, y0 - pad), min(H, y0 + sy + pad)
    rx0, rx1 = max(0, x0 - pad), min(W, x0 + sx + pad)
    dst_sel = np.zeros((H, W), bool); dst_sel[ry0:ry1, rx0:rx1] = True
    dst_sel &= (a < 0.10)
    if src_sel.sum() < 50 or dst_sel.sum() < 50:
        return patch
    p8 = (patch.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    h8 = (host.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    p_lab = cv2.cvtColor(p8, cv2.COLOR_RGB2LAB).astype(np.float32)
    h_lab = cv2.cvtColor(h8, cv2.COLOR_RGB2LAB).astype(np.float32)
    sm, ss = p_lab[src_sel].mean(0), p_lab[src_sel].std(0) + 1e-6
    dm, ds = h_lab[dst_sel].mean(0), h_lab[dst_sel].std(0) + 1e-6
    gain = np.clip(ds / ss, *std_clip)
    out = np.clip((p_lab - sm) * gain + dm, [0, 0, 0], [255, 255, 255]).astype(np.uint8)
    rgb = cv2.cvtColor(out, cv2.COLOR_LAB2RGB)
    return torch.from_numpy(rgb.astype(np.float32) / 255.0).permute(2, 0, 1)


# --- Mode B: MRSP sets size, orientation, anchor and containment --------------------
def _mask_centroid(a):
    m = a.numpy() > 0.5
    if m.sum() == 0:
        return None
    ys, xs = np.nonzero(m)
    return float(ys.mean()), float(xs.mean())

def _principal_angle(binary):
    """major-axis angle in degrees from second central moments; area comes free as m00"""
    m = cv2.moments(binary.astype(np.uint8), binaryImage=True)
    if m["m00"] <= 0:
        return None, 0.0
    mu20, mu11, mu02 = m["mu20"] / m["m00"], m["mu11"] / m["m00"], m["mu02"] / m["m00"]
    if abs(mu20 - mu02) < 1e-9 and abs(mu11) < 1e-9:
        return 0.0, m["m00"]
    return math.degrees(0.5 * math.atan2(2.0 * mu11, mu20 - mu02)), m["m00"]

def place_alpha_only(host_shape, entry, center, long_px, theta_deg, flip):
    """The alpha half of place_entry, without the RGB half.

    place_mode_B calls place_entry up to 25 times per sample and throws the RGB patch away every
    time -- it only ever reads the returned alpha. Each of those calls was cloning the 3x1024x1024
    host (12 MB) and resizing + warping the crop's RGB for nothing. This does the identical alpha
    arithmetic and skips both, which is where most of mode B's cost was.

    Output is bit-identical to place_entry(...)[1] with region=ones: same resize interpolation,
    same padding, same rotation matrix, same clipping.
    """
    H, W = host_shape
    a = entry["alpha"]
    if flip:
        a = a[:, ::-1].copy()

    r = long_px / max(entry["defect_long_px"], 1)
    s = int(np.clip(round(entry["side"] * r), 8, max(H, W)))
    a = cv2.resize(a, (s, s), interpolation=cv2.INTER_LINEAR)

    if theta_deg:
        p = int(math.ceil(s * (math.sqrt(2) - 1) / 2)) + 2
        a = cv2.copyMakeBorder(a, p, p, p, p, cv2.BORDER_CONSTANT, value=0)
        s2 = a.shape[0]
        M = cv2.getRotationMatrix2D((s2 / 2, s2 / 2), theta_deg, 1.0)
        a = cv2.warpAffine(a, M, (s2, s2), flags=cv2.INTER_LINEAR, borderValue=0)
        s = s2

    cy, cx = center
    y0 = int(np.clip(cy - s // 2, 0, max(0, H - s))); x0 = int(np.clip(cx - s // 2, 0, max(0, W - s)))
    y1, x1 = min(H, y0 + s), min(W, x0 + s)
    sy, sx = y1 - y0, x1 - x0

    alpha = torch.zeros(H, W)
    alpha[y0:y1, x0:x1] = torch.from_numpy(np.clip(a[:sy, :sx], 0, 1).astype(np.float32))
    return alpha


def place_mode_B(host, entry, blob, region, bc, long_px, theta, flip, gamma):
    """-> (centre, long_px, theta, containment). Falls back to mode C's arguments if degenerate."""
    _, H, W = host.shape
    bf = (blob.numpy() > 0.5).astype(np.float32)
    phi_b, area_b = _principal_angle(bf)
    af0 = place_alpha_only((H, W), entry, bc, long_px, 0.0, flip)
    phi_m, area_m = _principal_angle((af0.numpy() > 0.5).astype(np.uint8))
    if phi_b is None or phi_m is None or area_m <= 0 or area_b <= 0:
        return bc, long_px, theta, float("nan")

    # SIZE: the defect takes `gamma` of the blob; the rest is margin the collar can use.
    s = float(np.clip(math.sqrt(max(gamma * area_b, 1.0) / area_m), *GAMMA_S_RANGE))

    # ANCHOR: mask centroid on blob centroid. A bent sliver (can: 322x39) has its centroid OFF
    # its own mask, and a ragged blob can have its centroid outside itself -- in either case use
    # the deepest interior point, which is guaranteed to be inside.
    byi, bxi = int(np.clip(bc[0], 0, H - 1)), int(np.clip(bc[1], 0, W - 1))
    if bf[byi, bxi] > 0.5:
        anchor = (float(bc[0]), float(bc[1]))
    else:
        dt = cv2.distanceTransform((bf > 0.5).astype(np.uint8), cv2.DIST_L2, 5)
        yx = np.unravel_index(int(np.argmax(dt)), dt.shape)
        anchor = (float(yx[0]), float(yx[1]))

    d = phi_b - phi_m                       # ORIENTATION: align the two principal axes
    best = None
    for _ in range(3):                      # CONTAINMENT: shrink until it fits
        lp = long_px * s
        for th in (d, -d, d + 180.0, -d + 180.0):
            th %= 360.0
            afp = place_alpha_only((H, W), entry, (H // 2, W // 2), lp, th, flip)
            mc = _mask_centroid(afp)
            if mc is None:
                continue
            mb = (afp.numpy() > 0.5).astype(np.uint8)
            ay, ax = mc
            if mb[int(np.clip(ay, 0, H - 1)), int(np.clip(ax, 0, W - 1))] == 0:
                mdt = cv2.distanceTransform(mb, cv2.DIST_L2, 5)
                ay, ax = np.unravel_index(int(np.argmax(mdt)), mdt.shape)
            c = (int(H // 2 + (anchor[0] - ay)), int(W // 2 + (anchor[1] - ax)))
            af = place_alpha_only((H, W), entry, c, lp, th, flip)
            mm = af.numpy() > 0.5
            ct = float((mm & (bf > 0.5)).sum()) / max(mm.sum(), 1)
            if best is None or ct > best[0]:
                best = (ct, c, lp, th)
        if best and best[0] >= CONTAIN_T:
            break
        s = float(np.clip(s * 0.85, *GAMMA_S_RANGE))
    if best is None:
        return bc, long_px, theta, float("nan")
    ct, c, lp, th = best
    return c, lp, th, ct

print("placement + harmonisation ready (mode C + mode B).")

In [ ]:
# --- Compositing (verbatim from flash_part1) ----------------------------------------
def feather_mask(mask, sigma):
    return torch.from_numpy(cv2.GaussianBlur(mask.numpy().astype(np.float32), (0, 0), sigma)).clamp(0, 1)

def feather_sigma_for(mask, base_sigma=FEATHER_PX):
    """Cap the feather so it cannot fade the defect: a Gaussian with sigma comparable to the
    blob radius pulls the CORE below opacity 1.0, which is the bug this whole notebook avoids."""
    side, _ = mrsp_blob_stats(mask)
    if side <= 0:
        return float(base_sigma)
    return float(min(base_sigma, max(1.0, 0.25 * side)))

def alpha_composite(host, patch, alpha, region, beta=BETA, base_sigma=FEATHER_PX):
    sigma = feather_sigma_for(alpha, base_sigma)
    mf = (feather_mask(alpha, sigma) * region).unsqueeze(0)
    out = (host * (1 - mf) + (beta * patch + (1 - beta) * host) * mf).clamp(0, 1)
    # edit_mask = the region this operator actually modified. The seam metric is evaluated at
    # each arm's OWN boundary: the collar makes the Poisson arm edit a larger area, and scoring
    # both at the original mask would credit alpha for pixels it never touched.
    edit = ((mf[0] > 0.01).float().numpy() > 0).astype(np.uint8)
    return out, dict(sigma=sigma, edit_mask=edit)


def lab_of(t):
    return cv2.cvtColor((t.permute(1, 2, 0).numpy() * 255).astype(np.uint8),
                        cv2.COLOR_RGB2LAB).astype(np.float32)

def defect_thresholds(host, patch, mask_bin, ring_px=4):
    """Per-composite Lab cutoffs separating "defect" from "substrate".

    A fixed cutoff does not survive contact with a granular category. On rice or walnuts the
    patch's grains never line up with the host's, so ||patch - host|| is large across the whole
    crop and a fixed 6-unit cutoff calls the entire substrate a defect. The cutoffs are
    therefore taken from the composite's own distribution: the substrate noise floor is
    measured on the ring OUTSIDE the mask, where by definition there is no defect, and the
    defect cutoff is placed above it.
    """
    k = np.ones((3, 3), np.uint8)
    m = (mask_bin.numpy() > 0.5).astype(np.uint8)
    outer = (cv2.dilate(m, k, iterations=3 * ring_px) - cv2.dilate(m, k, iterations=ring_px)) > 0
    d = np.linalg.norm(lab_of(patch) - lab_of(host), axis=2)
    if outer.sum() < 50:                       # degenerate placement; fall back to the constants
        return DEFECT_T, 4.0, float("nan")
    floor = float(np.quantile(d[outer], 0.90))  # what "same material, misaligned" looks like
    t_def = max(DEFECT_T, floor)                # defect must beat the misalignment floor
    t_sub = max(4.0, float(np.quantile(d[outer], 0.50)))
    return t_def, t_sub, floor

def defect_core(host, patch, mask_bin, T=None):
    """Pixels inside the mask that actually carry the defect. A recovered mask also contains a
    substrate margin, and the difference between the two is what the collar has to clear."""
    m = (mask_bin.numpy() > 0.5).astype(np.uint8)
    if T is None:
        T, _, _ = defect_thresholds(host, patch, mask_bin)
    d = np.linalg.norm(lab_of(patch) - lab_of(host), axis=2)
    k = np.ones((3, 3), np.uint8)
    c = ((d > T) & (m > 0)).astype(np.uint8)
    c = cv2.morphologyEx(c, cv2.MORPH_OPEN, k, iterations=1)
    return cv2.morphologyEx(c, cv2.MORPH_CLOSE, k, iterations=2)

def collar_for(core_np, frac=COLLAR_FRAC, min_px=COLLAR_MIN_PX, keep=COLLAR_KEEP_R):
    """Collar width in px, from the defect's equivalent radius. frac=0 means NO collar, and
    the floor does not apply -- otherwise the control sweep could never reach the vanilla
    NORMAL_CLONE it exists to compare against.

    COLLAR_MIN_PX is a floor meant to HELP small defects get a workable boundary, but on a
    defect smaller than ~2.5x the floor it does the opposite: a 6px collar on a defect of
    equivalent radius 9.7px takes 62% of the radius, and NORMAL_CLONE then rebuilds what is
    left from substrate. The cap keeps at least `keep` of the radius as defect, so the floor
    can never be the thing that dissolves the defect it was added to protect.
    """
    if frac <= 0:
        return 0
    a = float(core_np.sum())
    if a < 1:
        return int(min_px)
    r_eq = math.sqrt(a / math.pi)
    w = max(min_px, round(frac * r_eq))
    return int(max(1, min(w, math.floor((1.0 - keep) * r_eq))))

def poisson_clone(host, patch, mask_np):
    """cv2 NORMAL_CLONE on an explicit mask. Returns None if the mask is unusable."""
    mk = (mask_np > 0).astype(np.uint8) * 255
    mk[:3, :] = 0; mk[-3:, :] = 0; mk[:, :3] = 0; mk[:, -3:] = 0
    ys, xs = np.where(mk > 0)
    if len(xs) < 10:
        return None
    c = (int((xs.min() + xs.max()) / 2), int((ys.min() + ys.max()) / 2))
    d = (host.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    s = (patch.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    try:
        out = cv2.seamlessClone(s, d, mk, c, cv2.NORMAL_CLONE)
    except cv2.error:
        return None
    return torch.from_numpy(out.astype(np.float32) / 255.0).permute(2, 0, 1).clamp(0, 1)

def collar_poisson(host, patch, alpha, region, frac=COLLAR_FRAC, min_px=COLLAR_MIN_PX):
    """Dilate the defect mask by a substrate collar, then solve. The collar moves the Dirichlet
    boundary off the defect and onto substrate, which is the whole mechanism."""
    hard = ((alpha > 0.5).float() * region)
    core_np = defect_core(host, patch, hard)
    if core_np.sum() < 10:                       # low-contrast defect: no content core to clear
        core_np = (hard.numpy() > 0.5).astype(np.uint8)
    w = collar_for(core_np, frac, min_px)
    k = np.ones((3, 3), np.uint8)
    comp = cv2.dilate((hard.numpy() > 0.5).astype(np.uint8), k, iterations=w) if w > 0 \
        else (hard.numpy() > 0.5).astype(np.uint8)
    comp = (comp * (region.numpy() > 0.5)).astype(np.uint8)   # never composite off-object
    info = dict(collar_px=w, core_px=int(core_np.sum()), comp_px=int(comp.sum()), fallback=None)

    info["edit_mask"] = comp
    out = poisson_clone(host, patch, comp)
    if out is None:
        info["fallback"] = "alpha (seamlessClone declined the mask)"
        out, ai = alpha_composite(host, patch, alpha, region)
        info["edit_mask"] = ai["edit_mask"]
    return out, info

def composite(arm, host, patch, alpha, region, **kw):
    if arm == "alpha":
        return alpha_composite(host, patch, alpha, region, **kw)
    if arm == "poisson":
        return collar_poisson(host, patch, alpha, region, **kw)
    raise KeyError(arm)

print("compositing operators ready.")

In [ ]:
# --- One synthesis call (from flash_part1, evaluation fields removed) ---------------
def synthesize(cat, host_path, entry, seed=SEED, arm="poisson", collar_frac=COLLAR_FRAC):
    """I_h + entry -> I_syn, M_gt. Returns None when the sample cannot be made.

    The RNG draws that pick the blob, the rotation, the flip and the C/B mode depend ONLY on
    the key string, so the corpus is reproducible: same seed -> same image, byte for byte."""
    host, region0 = host_and_region(host_path)
    _, H, W = host.shape
    if float(region0.mean()) < MIN_OBJ_COV_CAT[cat]:
        return None
    region, blob = mrsp_mask_for(host, seed=seed, region=region0)
    if blob.sum() < MIN_BLOB_PX:
        return None
    _, centre = mrsp_blob_stats(blob)
    if centre is None:
        return None

    # crc32 of the key string, NOT hash(): str hashing is salted per process unless
    # PYTHONHASHSEED is set, so hash() would give a different corpus on every session.
    key = f"{seed}|{cat}|{os.path.basename(host_path)}|{entry['key']}".encode()
    g = random.Random(zlib.crc32(key))
    theta = g.uniform(0, 360) if ROTATE else 0.0
    flip  = g.random() < 0.5
    long_px = target_long_px(entry, H, W)

    mode = "B" if g.random() < PLACEMENT_MIX else "C"
    gamma = g.uniform(*GAMMA_RANGE)
    if mode == "B":
        centre, long_px, theta, contain = place_mode_B(host, entry, blob, region, centre,
                                                       long_px, theta, flip, gamma)
    else:
        contain = float("nan")

    patch, alpha, box = place_entry(host, entry, centre, long_px, theta, flip, region)
    if (alpha > 0.5).sum() < 20:
        return None
    ph = harmonise(patch, alpha, host, box)

    # Below R_EQ_POISSON the collar takes more of the defect than it can spare and NORMAL_CLONE
    # rebuilds the interior from substrate -- a dissolved sample is a normal image carrying a
    # defect label. Those composite with alpha instead. MODE B ONLY.
    area_px = float((alpha > 0.5).float().sum())
    r_eq_px = math.sqrt(area_px / math.pi)
    use_arm = arm
    if mode == "B" and arm == "poisson" and r_eq_px < R_EQ_POISSON:
        use_arm = "alpha"
    out, info = composite(use_arm, host, ph, alpha, region,
                          **({"frac": collar_frac} if use_arm == "poisson" else {}))

    # GT is the warped DEFECT mask. Never the MRSP blob, and never the dilated collar mask:
    # the collar is a compositing device, and writing it to GT would inflate every target.
    gt = ((alpha > 0.5).float() * region)
    return dict(cat=cat, host_path=host_path, entry=entry["key"], seed=seed,
                mode=mode, gamma=gamma if mode == "B" else float("nan"),
                theta=theta, flip=flip, contain=contain, r_eq=r_eq_px,
                arm_used=use_arm, collar_px=info.get("collar_px", 0),
                blob_px=float(blob.sum()), gt_px=float((gt > 0.5).sum()),
                # host is returned because d_in measures |out - host| inside the GT, which is
                # the one number that says whether the composite carries a findable defect.
                host=host, out=out, gt=gt)

print("synthesize ready.")

## 6. Synthesise

Generates the corpus. Progress prints every ten images with an elapsed time and an estimate.

Hosts are shared across seeds, so host cost does not grow with the number of seeds. The seed varies
the placement blob, the bank entry and the rotation.

The loop walks the host pool until `N_SYNTH` images are made rather than taking the first
`N_SYNTH` hosts, because a host occasionally yields no usable placement.

In [ ]:
# Which bank entry a given (seed, attempt) retrieves. Verbatim from flash_part1 -- the 7 keeps
# consecutive hosts from all drawing the same entry.
def _entry_for(cat, seed, idx):
    return BANK[cat][(seed * 7 + idx) % len(BANK[cat])]


SYN_DIR = FINAL_DIR
# Clear first. Your last run died mid-loop after writing one image; without this a re-run in the
# same session mixes the two corpora and the manifest disagrees with what is on disk.
if os.path.isdir(SYN_DIR):
    shutil.rmtree(SYN_DIR)
os.makedirs(SYN_DIR, exist_ok=True)

CORPUS = {}          # (cat, seed) -> list of records
t0 = time.time()
n_done, n_target = 0, sum(len(SEEDS) * N_SYNTH for _ in LIVE_CATS)

for cat in LIVE_CATS:
    for seed in SEEDS:
        set_seed(1000 * seed + 7)
        hosts = list(HOST_PATHS[cat])
        random.Random(SEED).shuffle(hosts)   # ONE fixed permutation, identical for every seed
        CORPUS[(cat, seed)] = []

        made, ai = 0, -1
        for ai, hp in enumerate(hosts):
            if made >= N_SYNTH:
                break
            e = _entry_for(cat, seed, ai)
            _ts = time.time()
            d = synthesize(cat, hp, e, seed=1000 * seed + ai)
            _secs = time.time() - _ts
            if d is None:
                continue                      # empty frame / degenerate blob / tiny alpha

            sub = os.path.join(SYN_DIR, cat, f"seed{seed}")
            os.makedirs(sub, exist_ok=True)
            stem = f"{made:03d}_{d['entry']}"
            ip = os.path.join(sub, stem + ".png")
            mp = os.path.join(sub, stem + "_mask.png")
            Image.fromarray((as_np(d["out"]) * 255).astype(np.uint8)).save(ip)
            Image.fromarray(((d["gt"].numpy() > 0.5) * 255).astype(np.uint8)).save(mp)

            # d_in: how much the composite actually differs from the host INSIDE the ground
            # truth, 0-255. This is the one number that says whether the sample is real -- near
            # zero means the GT marks a region identical to the host, so a detector would be
            # scored on finding a defect that is not in the image.
            _g = (d["gt"] > 0.5).numpy()
            _dif = np.abs(as_np(d["out"]) - as_np(d["host"])).mean(2) * 255.0
            d_in = float(_dif[_g].mean()) if _g.any() else 0.0

            CORPUS[(cat, seed)].append(dict(
                img=ip, mask=mp, host=os.path.basename(hp), entry=d["entry"], d_in=d_in,
                gt_frac=float((d["gt"] > 0.5).float().mean()),
                mode=d["mode"], gamma=d["gamma"], contain=d["contain"], r_eq=d["r_eq"],
                blob_px=d["blob_px"], arm_used=d["arm_used"],
                collar=d["collar_px"], secs=_secs))
            made += 1
            n_done += 1
            if n_done % 10 == 0 or n_done == 1:
                el = time.time() - t0
                print(f"    {n_done:4d}/{n_target}  {el:7.1f}s  {el/n_done:6.2f}s/img"
                      f"  eta {(n_target-n_done)*el/n_done:7.1f}s")

        recs = CORPUS[(cat, seed)]
        short = "" if made >= N_SYNTH else \
                f"   SHORT: {made}/{N_SYNTH} after exhausting all {len(hosts)} hosts"
        nb = sum(1 for r in recs if r["mode"] == "B")
        nal = sum(1 for r in recs if r["arm_used"] == "alpha")
        print(f"  {cat:12s} seed {seed}: {len(recs):3d} images   modeB={nb}/{len(recs)}"
              f"  routed_to_alpha={nal}   ({ai + 1} hosts tried){short}")

SYNTH_SECONDS = time.time() - t0
n_img = sum(len(v) for v in CORPUS.values())
print(f"\n{n_img} synthetic images in {SYNTH_SECONDS:.0f}s -> {SYN_DIR}")

## 7. Manifest and checks

Writes `manifest.csv` and prints per-category statistics.

`d_in` is the mean difference between the composite and the host inside the ground truth, on a 0 to
255 scale. A sample below `DISSOLVED_T` carries no findable defect: the label marks a region the
blend left unchanged.

Those samples are counted, not removed. If one placement mode or one category dominates the count,
that is worth investigating before changing any threshold.

In [ ]:
MANI = pd.DataFrame([dict(cat=c, seed=s, **r) for (c, s), v in CORPUS.items() for r in v])
if MANI.empty:
    raise SystemExit("no images were synthesised -- see the per-category lines above")
MANI["dissolved"] = MANI.d_in < DISSOLVED_T
MANI.to_csv(os.path.join(SYN_DIR, "manifest.csv"), index=False)

json.dump(dict(seed=SEED, bank_root=BANK_ROOT, ad2_root=AD2_ROOT,
               cats=LIVE_CATS, seeds=SEEDS, n_synth=N_SYNTH, n_images=int(n_img),
               work_size=WORK_SIZE, coverage=AD2_COVERAGE, noise_scale=AD2_NOISE_SCALE,
               octaves=OCTAVES_FBM, persistence=PERSISTENCE, mrsp_alpha=MRSP_ALPHA,
               placement_mix=PLACEMENT_MIX, gamma_range=list(GAMMA_RANGE),
               collar_frac=COLLAR_FRAC, r_eq_poisson=R_EQ_POISSON, dissolved_t=DISSOLVED_T,
               mrsp_device=MRSP_DEVICE, synth_seconds=round(SYNTH_SECONDS, 1)),
          open(os.path.join(SYN_DIR, "run_config.json"), "w"), indent=2)

print("per category:")
print(MANI.groupby("cat").agg(n=("d_in", "size"), d_in=("d_in", "mean"),
                              gt_frac=("gt_frac", "mean"), r_eq=("r_eq", "mean"),
                              dissolved=("dissolved", "sum")).round(3).to_string())

print(f"\nplacement mix: modeB={int((MANI['mode']=='B').sum())}/{len(MANI)}"
      f"  ({100*(MANI['mode']=='B').mean():.0f}%, target {100*PLACEMENT_MIX:.0f}%)")
print(f"routed to alpha: {int((MANI.arm_used=='alpha').sum())}/{len(MANI)}"
      f"  (mode B defects below r_eq {R_EQ_POISSON}px)")

n_bad = int(MANI.dissolved.sum())
if n_bad:
    print(f"\n{n_bad}/{len(MANI)} composites are dissolved (d_in < {DISSOLVED_T}): the ground "
          "truth marks a region the operator left identical to the host.")
    print(MANI[MANI.dissolved].groupby(["cat", "mode"]).size().to_string())
    print("  Kept on purpose. If one mode dominates this count, that is a result about the")
    print("  operator, not a bug to tune away.")
else:
    print(f"\nno dissolved composites (all d_in >= {DISSOLVED_T})")

In [ ]:
from IPython.display import FileLink, display

ZIP_PATH = os.path.join(OUT_ROOT, "Final Synthetic Images.zip")
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(SYN_DIR):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, os.path.relpath(full, OUT_ROOT))

print(f"{ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB, {n_img} images)")
display(FileLink(ZIP_PATH))

## 8. Timing

Per-category and per-image synthesis time, split by placement mode and by blending operator.

Adaptive placement runs a containment search and is more expensive than original placement. A
single average hides which of the two dominates.

Set `GEN_SECONDS` to your measured Module 1 cost before reading the comparison at the end of the
cell.

In [ ]:
_s = MANI.secs.tolist()
print(f"{'category':14s}{'images':>8s}{'total s':>10s}{'mean s':>9s}{'min s':>8s}{'max s':>8s}")
print("-" * 57)
for c in LIVE_CATS:
    ss = MANI[MANI.cat == c].secs.tolist()
    if ss:
        print(f"{c:14s}{len(ss):>8d}{sum(ss):>10.1f}{np.mean(ss):>9.3f}"
              f"{min(ss):>8.3f}{max(ss):>8.3f}")
print("-" * 57)
print(f"{'ALL':14s}{len(_s):>8d}{sum(_s):>10.1f}{np.mean(_s):>9.3f}"
      f"{min(_s):>8.3f}{max(_s):>8.3f}")

print(f"\nwall clock              {SYNTH_SECONDS:8.1f}s   (serial)")
print(f"per image (wall)        {SYNTH_SECONDS/max(n_img,1):8.3f}s")
print(f"MRSP device             {MRSP_DEVICE:>8s}")

for mode in ("C", "B"):
    ss = MANI[MANI["mode"] == mode].secs.tolist()
    if ss:
        print(f"  mode {mode:2s} {len(ss):4d} images  {np.mean(ss):.3f}s each"
              + ("   (containment search costs the difference)" if mode == "B" else ""))
for arm in ("poisson", "alpha"):
    ss = MANI[MANI.arm_used == arm].secs.tolist()
    if ss:
        print(f"  {arm:8s} {len(ss):4d} images  {np.mean(ss):.3f}s each")

GEN_SECONDS = 60.0      # per-call cost of the Stage 1 generator; set to YOUR measured value
print(f"\nAgainst per-sample generative synthesis at {GEN_SECONDS:.0f}s/image:")
print(f"  {n_img} images generatively  {n_img * GEN_SECONDS:10.0f}s")
print(f"  {n_img} images via FLASH 4-5 {SYNTH_SECONDS:10.1f}s")
print(f"  ratio                        {n_img * GEN_SECONDS / max(SYNTH_SECONDS, 1e-6):10.1f}x")
print("\nStages 1-3 are NOT in this ratio -- they are the fixed cost this notebook amortises.")
print("Quote the end-to-end number (Stages 1-3 + this) in the paper, not this ratio alone.")

## 9. Inspect the output

Shows a sample of the generated images with their masks outlined, read back from disk.

Three things to check.

The defect is on the product, not the background. If it is not, object detection failed for that
host.

There is no visible rectangular seam or halo around the defect. A visible seam is a shortcut
feature: a detector will learn it instead of the defect.

The defect is still present. Compare against the crop shown in section 2 if you are unsure.

In [ ]:
_per_cat = 3
for cat in LIVE_CATS:
    sel = MANI[MANI.cat == cat].head(_per_cat)
    if sel.empty:
        continue
    fig, ax = plt.subplots(2, len(sel), figsize=(4.2 * len(sel), 8.4), squeeze=False)
    for j, (_, r) in enumerate(sel.iterrows()):
        img = np.array(Image.open(r["img"]).convert("RGB"))
        m8 = (np.array(Image.open(r["mask"]).convert("L")) > 127).astype(np.uint8)
        ax[0, j].imshow(img)
        ax[0, j].set_title(f"{cat} · mode {r['mode']} · {r['arm_used']}\n"
                           f"r_eq {r['r_eq']:.0f}px · collar {r['collar']}px · "
                           f"d_in {r['d_in']:.0f}", fontsize=8)
        marked = img.copy()
        cont, _ = cv2.findContours(m8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(marked, cont, -1, (255, 0, 0), 3)
        ax[1, j].imshow(marked)
        ax[1, j].set_title(f"GT - {int(m8.sum())}px", fontsize=8)
        ax[0, j].axis("off"); ax[1, j].axis("off")
    plt.tight_layout(); plt.show()

## Before using this corpus

1. The placement mode split is close to `PLACEMENT_MIX`. A large skew means adaptive placement is
   failing its containment search and falling back.
2. Few samples routed to alpha. A large fraction means the bank's defects are mostly too small for
   Poisson blending, so the corpus does not exercise it.
3. The dissolved count is low.
4. The seam check in section 9 passes by eye. No metric here catches a shortcut feature.
5. `GEN_SECONDS` is your measured Module 1 cost, not the placeholder.